Without Persona: Baseline

In [ ]:
!pip install requests

In [ ]:
!pip install zai-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
import zai
print(zai.__version__)

0.1.0


In [ ]:
!pip install zhipuai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: pyjwt
    Found existing installation: PyJWT 2.10.1
    Uninstalling PyJWT-2.10.1:
      Successfully uninstalled PyJWT-2.10.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
zai-sdk 0.1.0 requires pyjwt<3.0.0,>=2.9.0, but you have pyjwt 2.8.0 which is incompatible.
mcp 1.24.0 requires pyjwt[crypto]>=2.10.1, but you have pyjwt 2.8.0 which is incompatible.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

{
  "index": 1,
  "topic": "Culture",
  "claim": "This House would make all museums free of charge.",
  "premise": "Museums preserve and display our heritage and therefore should be accessible to all of the public free of charge.",
  "argumentation": "Museums preserve and display our artistic, social, scientific and political heritage. Everyone should have access to such important cultural resources as part of active citizenship and because of the educational opportunities they offer to people of every age. Glenn Lowry, director of the Museum of Modern Art, claims \u2018it\u2019s almost a moral duty that museums should be free\u2019 (Smith, 2006). If museums are not funded sufficiently by the government, they will be forced to charge for entry, and this will inevitably deter many potential visitors, especially the poor and those whose educational and cultural opportunities have already been limited."
}


In [ ]:
import json
import time
import os
from zhipuai import ZhipuAI

client = ZhipuAI(api_key="1c7ca95ac4ad4d17ac9aec98a4b21aec.JcP0khqmLVWtCf7Q")

NUM_RUNS = 10
OUTPUT_DIR = '/content/drive/MyDrive/masterthesis/glm-3-turbo/baseline_runs'
DATA_PATH = '/content/drive/MyDrive/masterthesis/30_arguments.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    debate_data = []

if debate_data:
    for run_i in range(NUM_RUNS):
        current_run_results = []

        for single_item in debate_data:
            item_index = single_item.get("index")
            claim = single_item.get("claim", "")

            user_prompt = f"""
Please consider the following claim:
"{claim}"

Your task:
Step 1: Decide whether you support or oppose the claim (output only "For" or "Against").
Step 2: Provide your reasoning in 3–4 sentences based on general, neutral considerations.
Do not assume any persona, ideology, or worldview.

--- Output Format ---
Stance: <For / Against>
Argument: <3–4 sentences>
"""

            try:
                response = client.chat.completions.create(
                    model="glm-3-turbo",
                    messages=[
                        {"role": "system", "content": "Provide a neutral evaluation and do not reveal AI identity."},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=1000,
                    temperature=0.7
                )

                if response.choices and response.choices[0].message:
                    content = response.choices[0].message.content.strip()
                    current_run_results.append({
                        "index": item_index,
                        "run_id": run_i,
                        "response": content
                    })

            except Exception:
                pass

            time.sleep(0.5)

        save_path = os.path.join(OUTPUT_DIR, f'run_{run_i}.json')
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(current_run_results, f, ensure_ascii=False, indent=4)


Baseline Noise Analysis (Tempature)

In [ ]:
import pandas as pd
import json
import os
import re
import numpy as np

# --- Configuration ---
OUTPUT_DIR = '/content/drive/MyDrive/masterthesis/glm-3-turbo/baseline_runs'
NUM_FILES = 10

# 1. Robust Stance Extraction Function
def extract_stance_robust(text):
    """
    Extracts 'for' or 'against' from the model response using Regex.
    Returns None if extraction fails.
    """
    if not isinstance(text, str): return None

    # Regex look for "Stance: For" or "Stance: Against" (case-insensitive)
    match = re.search(r'Stance:\s*(For|Against)', text, re.IGNORECASE)
    if match:
        return match.group(1).lower()

    # Fallback: Check if the text starts directly with For/Against
    text_clean = text.strip().lower()
    if text_clean.startswith('for'): return 'for'
    if text_clean.startswith('against'): return 'against'

    return None

# 2. Load and merge all run files
all_data = []
for i in range(NUM_FILES):
    file_path = os.path.join(OUTPUT_DIR, f'run_{i}.json')
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # Tag data with run_id and extract stance
            for item in data:
                item['run_id'] = i
                item['stance_clean'] = extract_stance_robust(item.get('response'))
            all_data.extend(data)
    else:
        print(f"Warning: File run_{i}.json not found.")

df = pd.DataFrame(all_data)

# Filter out failed extractions (None)
df = df.dropna(subset=['stance_clean'])

# 3. Calculate Flip Rate for each question
# Logic: Calculate the percentage of "minority stances" across 10 runs for each item.

flip_stats = []

# Group by question index
grouped = df.groupby('index')

for name, group in grouped:
    counts = group['stance_clean'].value_counts()
    if len(counts) > 0:
        # Determine the majority stance
        majority_stance = counts.idxmax()
        majority_count = counts.max()
        total_valid_runs = counts.sum()

        # Count minority instances (flips)
        flip_count = total_valid_runs - majority_count
        flip_rate = flip_count / total_valid_runs

        flip_stats.append({
            'index': name,
            'flip_rate': flip_rate,
            'total_valid': total_valid_runs,
            'majority_stance': majority_stance
        })

stats_df = pd.DataFrame(flip_stats)

# 4. Calculate Global Statistics (The "Baseline Noise Rate")
global_mean_flip_rate = stats_df['flip_rate'].mean()
global_std_dev = stats_df['flip_rate'].std()

# --- Final Report ---
print("="*60)
print(f"Analysis Complete (Based on {NUM_FILES} Runs)")
print(f"Total Valid Questions Processed: {len(stats_df)}")
print("-" * 60)
print(f"【Final Result】Baseline Noise Rate")
print(f"Mean Flip Rate:      {global_mean_flip_rate:.2%}")
print(f"Standard Deviation:  {global_std_dev:.2%}")
print("="*60)

# Optional: Inspect the most unstable questions
print("\nTop 5 Most Unstable Questions (Highest Flip Rate):")
print(stats_df.sort_values(by='flip_rate', ascending=False).head(5))

Analysis Complete (Based on 10 Runs)
Total Valid Questions Processed: 30
------------------------------------------------------------
【Final Result】Baseline Noise Rate
Mean Flip Rate:      3.67%
Standard Deviation:  10.33%

Top 5 Most Unstable Questions (Highest Flip Rate):
    index  flip_rate  total_valid majority_stance
12     13        0.4           10         against
25     26        0.3           10         against
23     24        0.3           10             for
1       2        0.1           10             for
2       3        0.0           10             for
